In [1]:
import pandas as pd
from io import StringIO
import re

In [2]:
files = {
    "file1": "국기연_발간물_summary.csv",
    "file2": "국방부_정책자료_summary.csv",
    "file3": "국방위_국정감사_summary.csv",
    "file4": "국방위_보도_summary.csv",
    "file5": "국방위_회의록_summary.csv",
}

dfs = {}

for key, path in files.items():
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    dfs[key] = pd.DataFrame(lines, columns=["text"])

In [3]:
def clean_and_split(df):
    # 1️⃣ text 전체를 하나의 CSV 문자열로 합치기
    csv_text = "".join(df["text"].tolist())

    # 2️⃣ pandas로 CSV 재파싱
    new_df = pd.read_csv(StringIO(csv_text))

    return new_df

In [4]:
for key in files:
    dfs[key] = clean_and_split(dfs[key])

In [5]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return text

    # 1️⃣ 문두 쓰레기 제거
    text = re.sub(r'^[\[\(【]+', '', text)

    # 2️⃣ 문장 전체에서 불필요 문자 제거
    text = re.sub(r'[●◒∙○\[\]]', '', text)

    # 3️⃣ 공백 정리
    text = text.strip()

    return text


In [6]:
for key, df in dfs.items():
    df["clean_summary"] = df["clean_summary"].apply(clean_text)

In [7]:
dfs["file1"]['clean_summary'][1221]

"'능동 합성개구배열(SAA) 소나 신호처리 기술  (반영과제)  71 합성개구배열(SAA) 소나 기술 개발(응용) 311 대잠정찰용 무인잠수정 코어 기술 개발(선도형) 중장거리 은밀 수중음향 통신기술 잠수함에서 수중작전 수심과 속력을 유지하면서 수상함/육상 지휘소와 통신 및 정보교환이 가능한 부이형  저주파 수중통신 기술로, 도청 방지를 위한 은밀화와 다중경로 해결을 위한 등화기 기술을 포함 (반영과제)  54 잠수함용 장거리 은밀 수중통신 기술(응용) 청록 레이저를 이용한 수중 통신기술 해수투과율이 좋은 청록레이저를 이용한 수중통신 기술로, 잠수함이 잠항심도 유지 및 기동 중인 상태에서  육상지휘소와 실시간 접속하여 네트워크 중심전에 통합할 수 있는 새로운 개념의 초고속 통신기술 (반영과제)  63 청록 레이저를 이용한 수중 통신기술(응용) 복합재료 저소음 음향센서 기술 압전재-폴리머 복합재료 음향센서기술로, 자체소음 저감 구조설계 및 배열센서 최적화 설계 기술 등을 포함  (반영과제) 602 복합재료 저소음 음향센서 기술(응용/시험) 초세장형 선배열 예인소나 기술 예인음탐기 소형화와 경량화 설계를 통한 운용성 향상 및 탑재 중량 최소화를 위해 신호전처리/전송기의  마이크로 소형화 설계 및 제작기술을 활용한 초세장형 선배열 예인소나 설계기술 (반영과제)  66 초세장형 선배열 설계기술(응용) 026(cid:2)’17~’31 핵심기술기획서 일반본 핵심기술 과제반영도 형상적응배열 소나 기술 무인잠수정의 표면형상에 따라 배열형태의 소나센서를 설치해 배열이득과 고해상도 빔형성으로 탐지, 추적,  식별 능력을 향상시키는 기술 탐지/식별 자동화 기술 표적신호에서 추출된 능/수동 표적 특징인자 정보와 신경망 이론, 퍼지이론 등의 인공지능 식별기를 이용하여  수중표적을 식별하는 기술 (반영과제)  38 수중고속표적의 탐지/식별/추적 기술(응용) 무인체 통합 상황인식 기술 다중무인체로부터 획득된 정보의 융합을 통해 얻어진 정보를 기반으로 현재 광역 전장의 상황인식 및 전시

# -------------------------

In [ ]:
import pandas as pd
from io import StringIO
import re
import os

In [ ]:
def read_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def normalize_text(text: str) -> str:
    # 개행 → 공백
    text = re.sub(r'\n+', ' ', text)

    # 불필요한 특수 불릿 제거
    text = re.sub(r'[●◒∙○☞]', ' ', text)

    # 여러 공백 → 하나
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def split_to_sentences(text: str):
    # 온점(.) 기준 문장 분리
    #  -> 온점은 문장에 유지

    sentences = re.split(r'(?<=\.)\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def txt_to_sentence_df(path, source_name):
    raw = read_txt(path)
    clean = normalize_text(raw)
    sentences = split_to_sentences(clean)

    df = pd.DataFrame({
        "source": source_name,
        "sentence": sentences
    })

    return df

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

folders = {
    "국기연_발간물": os.path.join(BASE_DIR, "국기연_발간물"),
    "국방부_정책자료": os.path.join(BASE_DIR, "국방부_정책자료"),
    "국방위_국정감사": os.path.join(BASE_DIR, "국방위_국정감사"),
    "국방위_보도": os.path.join(BASE_DIR, "국방위_보도"),
    "국방위_회의록": os.path.join(BASE_DIR, "국방위_회의록"),
}

dfs = {}

for source, folder_path in folders.items():
    sentence_dfs = []

    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            full_path = os.path.join(folder_path, file)

            df = txt_to_sentence_df(
                path=full_path,
                source_name=source
            )

            df["file"] = file  # 🔥 어떤 txt에서 왔는지
            sentence_dfs.append(df)

    # 폴더 단위로 하나의 DF로 병합
    dfs[source] = pd.concat(sentence_dfs, ignore_index=True)

# -------------------------

In [5]:
import re
import pandas as pd
import os

In [6]:
def read_txt(path):
    with open(path, encoding="utf-8") as f:
        return f.read()
    
def remove_special_chars(text):
    # 공통 특수기호 제거
    text = re.sub(r'[●◒∙○■□◆▶]', ' ', text)
    text = re.sub(r'\(cid:\d+\)', ' ', text)      # (cid:xx)
    text = re.sub(r'-\d+%-', ' ', text)           # -3%-
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def split_sentences_da(text):
    """ '~다.' 기준 문장 분리 """
    return re.split(r'(?<=다\.)\s+', text)

In [7]:
def preprocess_kida(text):
    text = re.split(r'01\s*문서의\s*역할', text, maxsplit=1)[-1]
    text = remove_special_chars(text)
    sentences = re.split(r'(?<=\.)\s+', text)
    return sentences

def preprocess_mnd_policy(text):
    text = re.sub(r'\n+', ' ', text)
    text = remove_special_chars(text)
    return split_sentences_da(text)

def preprocess_audit(text):
    text = re.split(r'감\s*사\s*목\s*적|감사의\s*목적', text, maxsplit=1)[-1]
    text = remove_special_chars(text)

    # 특수기호 뒤를 문장 경계로 사용
    sentences = re.split(r'[●◒∙○]\s*', text)
    return sentences

def preprocess_press(text):
    # 배포일시 추출
    date_match = re.search(r'배포\s*일시[:：]\s*(.+)', text)
    date = date_match.group(1) if date_match else None

    # 본문 시작 컷
    text = re.split(r'국회|국방위', text, maxsplit=1)[-1]

    # 뒤쪽 불필요 제거
    text = re.split(r'\[사진|\【붙\s*임】', text)[0]

    text = remove_special_chars(text)
    sentences = split_sentences_da(text)

    return sentences, date

    
def preprocess_minutes(text):
    # 일시 추출
    date_match = re.search(r'\d{4}년\s*\d+월\s*\d+일', text)
    date = date_match.group(0) if date_match else None

    # 첫 ◯ 기준 컷
    text = re.split(r'◯', text, maxsplit=1)[-1]

    chunks = re.split(r'◯', text)
    chunks = [remove_special_chars(c) for c in chunks if len(c.strip()) > 20]

    return chunks, date


In [8]:
#공통 실행기
def process_txt(path, source):
    raw = read_txt(path)

    rows = []

    if source == "국기연_발간물":
        items = preprocess_kida(raw)

    elif source == "국방부_정책자료":
        items = preprocess_mnd_policy(raw)

    elif source == "국방위_국정감사":
        items = preprocess_audit(raw)

    elif source == "국방위_보도":
        items, date = preprocess_press(raw)

    elif source == "국방위_회의록":
        items, date = preprocess_minutes(raw)

    else:
        raise ValueError("Unknown source")

    for item in items:
        if len(item.strip()) > 15:
            rows.append({
                "source": source,
                "text": item.strip()
            })

    return pd.DataFrame(rows)


In [12]:
BASE_DIR = "./변환됨"   # 현재 작업 디렉토리
OUTPUT_DIR = "정돈된_"
os.makedirs(OUTPUT_DIR, exist_ok=True)

folders = {
    "국기연_발간물": "국기연_발간물",
    "국방부_정책자료": "국방부_정책자료",
    "국방위_국정감사": "국방위_국정감사",
    "국방위_보도": "국방위_보도",
    "국방위_회의록": "국방위_회의록",
}

all_dfs = {}

for source, folder in folders.items():
    folder_path = os.path.join(BASE_DIR, folder)

    dfs = []

    for fname in os.listdir(folder_path):
        if not fname.endswith(".txt"):
            continue

        path = os.path.join(folder_path, fname)
        print(f"[처리 중] {source} / {fname}")

        df = process_txt(path, source)
        df["doc_name"] = fname
        df["chunk_id"] = range(len(df))

        dfs.append(df)

    if dfs:
        merged = pd.concat(dfs, ignore_index=True)
        all_dfs[source] = merged

        save_path = os.path.join(OUTPUT_DIR, f"{source}.csv")
        merged.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"✅ 저장 완료: {save_path}")


[처리 중] 국기연_발간물 / '21_'35 핵심기술기획서(일반본, DTiMS 탑재용).txt
[처리 중] 국기연_발간물 / '23-'37 국방기술기획서 일반본.txt
[처리 중] 국기연_발간물 / '24-'38 국방기술기획서(일반본).txt
[처리 중] 국기연_발간물 / '25-'39 국방기술기획서 일반본.txt
[처리 중] 국기연_발간물 / `17 ~ `31 핵심기술기획서 일반본.txt
[처리 중] 국기연_발간물 / ’18~’32 핵심기술기획서.txt
[처리 중] 국기연_발간물 / ’19~’33 핵심기술기획서.txt
[처리 중] 국기연_발간물 / 국방기술진흥연구소 - 22-36 국방기술기획서 일반본.txt
✅ 저장 완료: 정돈된_\국기연_발간물.csv
[처리 중] 국방부_정책자료 / 14-30.txt
[처리 중] 국방부_정책자료 / PBLICTNEBOOK_201411060459455410.txt
[처리 중] 국방부_정책자료 / PBLICTNEBOOK_201411060500327780.txt
[처리 중] 국방부_정책자료 / PBLICTNEBOOK_201411060501352120.txt
[처리 중] 국방부_정책자료 / PBLICTNEBOOK_202303090216140530.txt
[처리 중] 국방부_정책자료 / PBLICTNEBOOK_202303090232023320.txt
[처리 중] 국방부_정책자료 / PBLICTNEBOOK_202305021112384930.txt
[처리 중] 국방부_정책자료 / 국방개혁_2.0.txt
✅ 저장 완료: 정돈된_\국방부_정책자료.csv
[처리 중] 국방위_국정감사 / 02국감결과보고서.txt
[처리 중] 국방위_국정감사 / 03국감결과보고서[1].txt
[처리 중] 국방위_국정감사 / 04국감결과보고서(최종의결).txt
[처리 중] 국방위_국정감사 / 05국감결과보고서(최종)(05.11.29).txt
[처리 중] 국방위_국정감사 / 2000국감결과보고.txt
[처리 중] 국방위_국정감사 / 2001년도국감결과보고서.txt

In [24]:
import GEMINI_API
import google.generativeai as genai
import os
import pandas as pd
print(dir(GEMINI_API))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'call_gemini', 'genai', 'generate_content', 'model', 'os']


In [22]:
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

model = genai.GenerativeModel("models/gemini-1.5-flash")

In [21]:
def pack_chunks(chunks, max_chars=8000):
    """
    chunks: List[str]
    return: List[List[str]]
    """
    batches = []
    current = []
    current_len = 0

    for c in chunks:
        l = len(c)
        if current_len + l > max_chars:
            batches.append(current)
            current = [c]
            current_len = l
        else:
            current.append(c)
            current_len += l

    if current:
        batches.append(current)

    return batches

In [23]:
def call_gemini(prompt: str) -> str:
    response = model.generate_content(prompt)
    return response.text.strip()

def generate_content(chunks):
    joined = "\n".join(chunks)

    return f"""
다음은 대한민국 국회 국방위원회 회의록 일부이다.
정치적 수사는 모두 제거하고, 국방 연구과제·기술·결정사항 및 외국과의 기술 계약 중심으로 요약하라.

요약 조건:
- 핵심 쟁점 3~5개
- 국방 연구과제·기술·결정사항 및 외국과의 기술 계약 관련 내용 우선
- 추측이나 해석 금지

회의록:
{joined}
"""

In [ ]:
from GEMINI_API import call_gemini

def summarize_one_minutes_doc(df_doc):
    chunks = df_doc["text"].tolist()

    batches = pack_chunks(chunks)
    partial_summaries = []

    # 1️⃣ Map 단계
    for i, batch in enumerate(batches):
        print(f"  - 부분 요약 {i+1}/{len(batches)}")
        prompt = generate_content(batch)
        summary = call_gemini(prompt)
        partial_summaries.append(summary)

    # 2️⃣ Reduce 단계
    final_prompt = f"""
다음은 국회 국방위원회 회의록의 부분 요약들이다.
이를 종합하여 최종 요약을 작성하라.

조건:
- 핵심 쟁점 5개 이내
- 국방 연구과제·기술·결정사항 및 외국과의 기술 계약 중심
- 중복 제거
- 불릿 포인트 형식

부분 요약:
{"\n\n".join(partial_summaries)}
"""

    final_summary = call_gemini(final_prompt)
    return final_summary

In [ ]:
minutes_df = pd.read_csv("processed_csv/국방위_회의록.csv")

results = []

for doc_name, g in minutes_df.groupby("doc_name"):
    print(f"\n[요약 중] {doc_name}")
    summary = summarize_one_minutes_doc(g)

    results.append({
        "doc_name": doc_name,
        "summary": summary
    })
